### Cell 1: import the tools

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### Cell 2: load the data

In [ ]:
df = pd.read_csv('../data/creditcard.csv')
df.shape

### Cell 3: look at the first rows

In [ ]:
df.head()

### Cell 4: how much fraud is there?

In [ ]:
df['Class'].value_counts()

### Cell 5: see the imbalance

In [ ]:
sns.countplot(x='Class', data=df)
plt.show()

### Cell 6: how are amounts spread out?

In [ ]:
sns.histplot(df['Amount'], bins=50)
plt.show()

### Cell 7: compare amounts for fraud and normal transactions

In [ ]:
df.groupby('Class')['Amount'].describe()

### Cell 8: sort by time and split

In [ ]:
df = df.sort_values('Time').reset_index(drop=True)
split_index = int(len(df) * 0.8)
train = df.iloc[:split_index].copy()
test = df.iloc[split_index:].copy()
print('Training rows:', len(train), '| fraud cases:', train['Class'].sum())
print('Test rows:', len(test), '| fraud cases:', test['Class'].sum())

### Cell 9: scale Amount and Time (corrected version)

In [ ]:
from sklearn.preprocessing import StandardScaler

amount_scaler = StandardScaler()
amount_scaler.fit(train[['Amount']])

time_scaler = StandardScaler()
time_scaler.fit(train[['Time']])

train['Amount_scaled'] = amount_scaler.transform(train[['Amount']])
train['Time_scaled'] = time_scaler.transform(train[['Time']])
test['Amount_scaled'] = amount_scaler.transform(test[['Amount']])
test['Time_scaled'] = time_scaler.transform(test[['Time']])

### Cell 10: separate the clues (X) from the answer (y)

In [ ]:
v_cols = [col for col in df.columns if col.startswith('V')]
feature_cols = v_cols + ['Amount_scaled', 'Time_scaled']

X_train = train[feature_cols]
y_train = train['Class']
X_test = test[feature_cols]
y_test = test['Class']

### Cell 11: train the model

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)

### Cell 12: predict and check the results

In [ ]:
from sklearn.metrics import classification_report

lr_pred = lr_model.predict(X_test)
print(classification_report(y_test, lr_pred))

### Cell 13: compute the curve and PR-AUC

In [ ]:
from sklearn.metrics import precision_recall_curve, auc

lr_scores = lr_model.predict_proba(X_test)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test, lr_scores)
lr_pr_auc = auc(recall, precision)
print('Logistic Regression PR-AUC:', round(lr_pr_auc, 4))

### Cell 14: draw the curve

In [ ]:
plt.plot(recall, precision, label='Logistic Regression')
plt.xlabel('Recall (share of fraud caught)')
plt.ylabel('Precision (share of alerts that are real fraud)')
plt.title('Precision-Recall Curve')
plt.legend()
plt.show()

### Cell 15: compare several thresholds

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

for threshold in [0.1, 0.2, 0.3, 0.5, 0.7]:
    predictions = (lr_scores >= threshold).astype(int)
    print(
        'Threshold', threshold,
        '| precision', round(precision_score(y_test, predictions, zero_division=0), 2),
        '| recall', round(recall_score(y_test, predictions), 2),
        '| F1', round(f1_score(y_test, predictions), 2),
        '| alerts', predictions.sum(),
    )

### Cell 16: train and evaluate XGBoost

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100, max_depth=4, eval_metric='logloss', random_state=42
)
xgb_model.fit(X_train, y_train)

xgb_scores = xgb_model.predict_proba(X_test)[:, 1]
xgb_pred = (xgb_scores >= 0.5).astype(int)
print(classification_report(y_test, xgb_pred))

p, r, _ = precision_recall_curve(y_test, xgb_scores)
xgb_pr_auc = auc(r, p)
print('XGBoost PR-AUC:', round(xgb_pr_auc, 4))

### Cell 17: try thresholds for XGBoost too

In [ ]:
for threshold in [0.1, 0.2, 0.3, 0.5, 0.7]:
    predictions = (xgb_scores >= threshold).astype(int)
    print(
        'Threshold', threshold,
        '| precision', round(precision_score(y_test, predictions, zero_division=0), 2),
        '| recall', round(recall_score(y_test, predictions), 2),
        '| alerts', predictions.sum(),
    )

### Cell 18: compare the two curves on one chart

In [ ]:
plt.plot(recall, precision, label=f'Logistic Regression (PR-AUC {lr_pr_auc:.2f})')
plt.plot(r, p, label=f'XGBoost (PR-AUC {xgb_pr_auc:.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Model comparison')
plt.legend()
plt.show()

### Cell 19: overall feature importance

In [ ]:
import shap

shap.initjs()
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test)

### Cell 20: find fraud cases the model caught and missed

In [ ]:
fraud_ids = y_test[y_test == 1].index[:10]

for idx in fraud_ids:
    probability = xgb_model.predict_proba(X_test.loc[[idx]])[0][1]
    print(idx, round(probability, 4))

### Cell 21: an explainer that speaks in probabilities

In [ ]:
background = X_train.sample(1000, random_state=42)
explainer_prob = shap.TreeExplainer(xgb_model, background, model_output='probability')

### Cell 22: explain a fraud the model caught

In [ ]:
caught_id = 229712
caught_values = explainer_prob.shap_values(X_test.loc[[caught_id]])
shap.force_plot(explainer_prob.expected_value, caught_values[0], X_test.loc[caught_id])

### Cell 23: explain a fraud the model missed

In [ ]:
missed_id = 231978
missed_values = explainer_prob.shap_values(X_test.loc[[missed_id]])
shap.force_plot(explainer_prob.expected_value, missed_values[0], X_test.loc[missed_id])

### Cell 24: save the model and its helpers

In [ ]:
import joblib

joblib.dump(xgb_model, '../src/fraud_model.pkl')
joblib.dump(feature_cols, '../src/feature_cols.pkl')
joblib.dump(amount_scaler, '../src/amount_scaler.pkl')
joblib.dump(time_scaler, '../src/time_scaler.pkl')

### Cell 25: save predictions for the dashboard

In [ ]:
THRESHOLD = 0.3

results = test.copy()
results['predicted_proba'] = xgb_model.predict_proba(X_test)[:, 1]
results['predicted_fraud'] = (results['predicted_proba'] >= THRESHOLD).astype(int)
results.to_csv('../data/dashboard_results.csv')
print('Flagged:', results['predicted_fraud'].sum())

### Cell 26: connect to OpenAI

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv('../.env')
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

### Cell 27: a function that explains any transaction

In [ ]:
def explain_in_plain_english(transaction_id):
    row = X_test.loc[[transaction_id]]
    probability = xgb_model.predict_proba(row)[0][1]
    values = explainer_prob.shap_values(row)[0]

    pairs = sorted(zip(feature_cols, values), key=lambda pair: abs(pair[1]), reverse=True)
    pairs = pairs[:5]
    top_features = '\n'.join(f'{name}: {value:+.4f}' for name, value in pairs)

    prompt = f"""A fraud detection model gave a transaction a fraud probability
of {probability:.2%}.

The five factors that most influenced this score were (positive pushes towards fraud,
negative pushes away from fraud):
{top_features}

Write a short, plain-English explanation (2-3 sentences) for a bank fraud analyst.

Requirements:
- Mention each factor by name with its value, and say whether it pushed the score up or down.
- These factors are anonymised transaction attributes. Do not guess or invent what they mean.
- Make clear this describes how the model behaved, not proof that the transaction is fraud.
- Do not use technical machine learning jargon.
"""

    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
    )
    return response.choices[0].message.content


print(explain_in_plain_english(229712))